# LTC -- Liquid Time-Constant Networks

Hasani, Lechner, Amini, Rus, Grosu, *Liquid Time-constant Networks*, AAAI 2021 ([arXiv:2006.04439](https://arxiv.org/abs/2006.04439)).

Each hidden unit obeys `dx/dt = -x/tau + f(x, I) * (A - x)`, where `f` is a sigmoidal synapse driven by the input -- the effective time constant changes with the input at every step. See `model.py` for the fused semi-implicit-Euler solver and `../../papers/README.md` for the reference.

This notebook trains an `LTCModel` on the UCI Ozone Level Detection dataset.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from liquid_playground.data import load_ozone
from liquid_playground.device import resolve_device
from liquid_playground.utils.seed import set_seed
from model import LTCModel

set_seed(0)
# device options: 'auto' (default, picks cuda/mps if available), 'cpu', 'cuda', 'mps'
device = resolve_device('auto')
print('device:', device)

In [ ]:
train_x, train_y, test_x, test_y = load_ozone()
train_x = train_x.unsqueeze(-1).to(device)
test_x = test_x.unsqueeze(-1).to(device)
train_y, test_y = train_y.to(device), test_y.to(device)
print(train_x.shape, train_y.shape)

In [ ]:
model = LTCModel(input_size=1, hidden_size=32, output_size=1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

history = {'train_loss': [], 'test_acc': []}
epochs = 40
for epoch in range(epochs):
    model.train()
    opt.zero_grad()
    logits = model(train_x).squeeze(-1)
    loss = loss_fn(logits, train_y)
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        test_acc = ((model(test_x).squeeze(-1) > 0).float() == test_y).float().mean().item()
    history['train_loss'].append(loss.item())
    history['test_acc'].append(test_acc)

print(f"final test accuracy: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_acc']); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()